# UK House Price Analysis (England & Wales)
**Data Source:** HM Land Registry - Price Paid Data (Open Government Licence)

**Author:** Kaveen Kodikarage

---

## Project Overview

This project analyses residential property sales across England and Wales using the HM Land Registry Price Paid dataset. The dataset is publicly available and updated monthly by the UK government.

### Key Questions Explored:
1. How have average house prices changed over time?
2. Which regions are the most and least affordable?
3. How does property type affect price?
4. What is the difference in price between new builds and existing properties?
5. Can we build a model to predict house prices from available features?

---

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Plot styling
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Blues_r')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.family'] = 'DejaVu Sans'

print('Libraries loaded successfully.')

## 2. Loading the Data

The HM Land Registry publishes Price Paid Data as open data. We load one year (2023) for analysis. You can swap the URL to load any year from 1995 onwards.

Column reference:
- **price** – Sale price in GBP
- **date** – Date of transfer
- **postcode** – Property postcode
- **property_type** – D=Detached, S=Semi-detached, T=Terraced, F=Flat/Maisonette, O=Other
- **new_build** – Y=New build, N=Existing property
- **tenure** – F=Freehold, L=Leasehold
- **town** – Town/City
- **county** – County

In [ ]:
# Column names as defined by HM Land Registry
columns = [
    'transaction_id', 'price', 'date', 'postcode',
    'property_type', 'new_build', 'tenure',
    'paon', 'saon', 'street', 'locality',
    'town', 'district', 'county', 'ppd_type', 'record_status'
]

# Load 2023 data directly from the Land Registry open data S3 bucket
url = 'http://prod.publicdata.landregistry.gov.uk.s3-website-eu-west-1.amazonaws.com/pp-2023.csv'
print('Downloading data from HM Land Registry...')
df_raw = pd.read_csv(url, header=None, names=columns, low_memory=False)
print(f'Loaded {len(df_raw):,} rows.')
df_raw.head(3)

## 3. Data Cleaning

In [ ]:
df = df_raw.copy()

# Parse date and extract useful time features
df['date'] = pd.to_datetime(df['date'])
df['year']  = df['date'].dt.year
df['month'] = df['date'].dt.month
df['month_name'] = df['date'].dt.strftime('%b')

# Keep only residential sales (exclude bulk transfers and other anomalies)
df = df[df['ppd_type'] == 'A']

# Drop rows with missing essential fields
df.dropna(subset=['price', 'county', 'property_type'], inplace=True)

# Remove extreme outliers (below £10k or above £10M — likely data errors or commercial)
df = df[(df['price'] >= 10_000) & (df['price'] <= 10_000_000)]

# Map property type codes to readable labels
property_map = {'D': 'Detached', 'S': 'Semi-Detached', 'T': 'Terraced', 'F': 'Flat', 'O': 'Other'}
df['property_label'] = df['property_type'].map(property_map)

# Map new build flag
df['new_build_label'] = df['new_build'].map({'Y': 'New Build', 'N': 'Existing Property'})

print(f'Clean dataset: {len(df):,} transactions')
print(f'Date range: {df["date"].min().date()} to {df["date"].max().date()}')
print(f'\nMissing values per column:')
print(df[['price','county','property_type','new_build','tenure']].isnull().sum())

In [ ]:
# Basic statistics
print('=== Price Summary Statistics (£) ===')
stats = df['price'].describe()
for label, val in stats.items():
    print(f'  {label:>10}: £{val:>12,.0f}')

## 4. Exploratory Data Analysis

### 4.1 Price Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of prices (capped at £1.5M for visibility)
plot_df = df[df['price'] <= 1_500_000]
axes[0].hist(plot_df['price'], bins=80, color='#1B3A6B', edgecolor='white', linewidth=0.4)
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1000:.0f}k'))
axes[0].set_title('Distribution of Sale Prices (up to £1.5M)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Sale Price')
axes[0].set_ylabel('Number of Transactions')

# Log-scale version to show full spread
axes[1].hist(np.log10(df['price']), bins=60, color='#2E5FA3', edgecolor='white', linewidth=0.4)
axes[1].set_title('Log-Scale Price Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('log₁₀(Sale Price)')
axes[1].set_ylabel('Number of Transactions')
tick_vals = [4, 4.5, 5, 5.5, 6, 6.5]
axes[1].set_xticks(tick_vals)
axes[1].set_xticklabels([f'£{10**v:,.0f}' for v in tick_vals], rotation=30)

plt.suptitle('England & Wales House Price Distribution — 2023', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('price_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Median price: £{df["price"].median():,.0f}')
print(f'Mean price:   £{df["price"].mean():,.0f}')

### 4.2 Transaction Volume by Month

In [ ]:
monthly = df.groupby('month').agg(
    transactions=('price', 'count'),
    avg_price=('price', 'median')
).reset_index()

month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()

bars = ax1.bar(monthly['month'], monthly['transactions'], color='#1B3A6B', alpha=0.8, label='Transactions')
line = ax2.plot(monthly['month'], monthly['avg_price'], color='#E87722', linewidth=2.5,
                marker='o', markersize=6, label='Median Price')

ax1.set_xticks(range(1,13))
ax1.set_xticklabels(month_labels)
ax1.set_ylabel('Number of Transactions', color='#1B3A6B')
ax2.set_ylabel('Median Sale Price (£)', color='#E87722')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1000:.0f}k'))

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.title('Monthly Property Transactions & Median Price — England & Wales 2023',
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('monthly_volume.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.3 Average Price by Property Type

In [ ]:
prop_stats = df[df['property_label'] != 'Other'].groupby('property_label').agg(
    median_price=('price', 'median'),
    count=('price', 'count')
).sort_values('median_price', ascending=False).reset_index()

colours = ['#1B3A6B', '#2E5FA3', '#4A80C4', '#7AAAD8']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(prop_stats['property_label'], prop_stats['median_price'],
               color=colours, edgecolor='white')

for bar, (_, row) in zip(bars, prop_stats.iterrows()):
    ax.text(bar.get_width() + 2000, bar.get_y() + bar.get_height()/2,
            f'£{row["median_price"]:,.0f}  ({row["count"]:,} sales)',
            va='center', fontsize=10)

ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1000:.0f}k'))
ax.set_xlabel('Median Sale Price')
ax.set_title('Median Sale Price by Property Type — England & Wales 2023',
             fontsize=13, fontweight='bold')
ax.set_xlim(0, prop_stats['median_price'].max() * 1.3)
plt.tight_layout()
plt.savefig('price_by_type.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.4 New Build vs Existing Property

In [ ]:
nb_stats = df[df['property_label'] != 'Other'].groupby(
    ['property_label', 'new_build_label'])['price'].median().reset_index()

pivot = nb_stats.pivot(index='property_label', columns='new_build_label', values='price')
pivot = pivot.sort_values('Existing Property', ascending=False)

ax = pivot.plot(kind='bar', color=['#1B3A6B', '#7AAAD8'], figsize=(11, 6),
                edgecolor='white', width=0.7)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1000:.0f}k'))
ax.set_xticklabels(pivot.index, rotation=0)
ax.set_ylabel('Median Sale Price')
ax.set_xlabel('')
ax.set_title('New Build vs Existing Property Prices by Type — 2023',
             fontsize=13, fontweight='bold')
ax.legend(title='')

# Annotate with premium %
for i, ptype in enumerate(pivot.index):
    if 'New Build' in pivot.columns and 'Existing Property' in pivot.columns:
        nb_val = pivot.loc[ptype, 'New Build']
        ex_val = pivot.loc[ptype, 'Existing Property']
        if pd.notna(nb_val) and pd.notna(ex_val):
            premium = ((nb_val - ex_val) / ex_val) * 100
            ax.text(i, max(nb_val, ex_val) + 5000, f'+{premium:.0f}% NB',
                    ha='center', fontsize=9, color='#1B3A6B', fontweight='bold')

plt.tight_layout()
plt.savefig('new_vs_existing.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.5 Top 15 Most & Least Expensive Counties

In [ ]:
county_stats = df.groupby('county').agg(
    median_price=('price', 'median'),
    count=('price', 'count')
).reset_index()

# Only include counties with enough sales for statistical reliability
county_stats = county_stats[county_stats['count'] >= 100]

top15    = county_stats.nlargest(15, 'median_price')
bottom15 = county_stats.nsmallest(15, 'median_price')

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Most expensive
axes[0].barh(top15['county'][::-1], top15['median_price'][::-1],
             color='#1B3A6B', edgecolor='white')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1000:.0f}k'))
axes[0].set_title('15 Most Expensive Counties', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Median Sale Price')

# Least expensive
axes[1].barh(bottom15['county'], bottom15['median_price'],
             color='#7AAAD8', edgecolor='white')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1000:.0f}k'))
axes[1].set_title('15 Most Affordable Counties', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Median Sale Price')

plt.suptitle('County-Level House Price Comparison — England & Wales 2023',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('county_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 5 Most Expensive:')
for _, row in top15.head(5).iterrows():
    print(f"  {row['county']:<30} £{row['median_price']:>10,.0f}")

print('\nTop 5 Most Affordable:')
for _, row in bottom15.head(5).iterrows():
    print(f"  {row['county']:<30} £{row['median_price']:>10,.0f}")

### 4.6 Freehold vs Leasehold

In [ ]:
tenure_map = {'F': 'Freehold', 'L': 'Leasehold'}
tenure_df = df[df['tenure'].isin(['F','L'])].copy()
tenure_df['tenure_label'] = tenure_df['tenure'].map(tenure_map)

tenure_stats = tenure_df.groupby('tenure_label')['price'].describe()[['count','mean','50%']]
tenure_stats.columns = ['Count', 'Mean Price', 'Median Price']
tenure_stats['Count'] = tenure_stats['Count'].astype(int)
print(tenure_stats.to_string())

fig, ax = plt.subplots(figsize=(8, 5))
tenure_df.boxplot(column='price', by='tenure_label', ax=ax,
                  flierprops={'marker': '.', 'markersize': 2, 'alpha': 0.3},
                  patch_artist=True)
ax.set_ylim(0, 1_200_000)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1000:.0f}k'))
ax.set_title('Price Distribution: Freehold vs Leasehold — 2023', fontsize=12, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Sale Price')
plt.suptitle('')
plt.tight_layout()
plt.savefig('freehold_vs_leasehold.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Predictive Modelling — Linear Regression

We build a simple linear regression model to predict house prices based on property type, tenure, new build status, and month of sale. This is a baseline model — in practice, location (postcode level) and property size would be the strongest predictors.

**Features used:**
- Property type (encoded)
- Tenure type (encoded)
- New build flag (encoded)
- Month of sale

In [ ]:
# Prepare features
model_df = df[df['property_label'] != 'Other'].dropna(
    subset=['property_type', 'tenure', 'new_build', 'month']
).copy()

le = LabelEncoder()
model_df['property_enc'] = le.fit_transform(model_df['property_type'])
model_df['tenure_enc']   = le.fit_transform(model_df['tenure'])
model_df['new_build_enc']= le.fit_transform(model_df['new_build'])

features = ['property_enc', 'tenure_enc', 'new_build_enc', 'month']
X = model_df[features]
y = model_df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2  = r2_score(y_test, y_pred)

print('=== Model Performance ===')
print(f'  Mean Absolute Error : £{mae:,.0f}')
print(f'  R² Score            : {r2:.4f}')
print()
print('Note: Low R² is expected — property type and tenure alone explain a small')
print('portion of price variance. Location data (postcode) would significantly')
print('improve predictions in a more advanced model.')

In [ ]:
# Visualise actual vs predicted (sample of 1000 for clarity)
sample_idx = np.random.choice(len(y_test), size=min(1000, len(y_test)), replace=False)
y_test_s = np.array(y_test)[sample_idx]
y_pred_s = y_pred[sample_idx]

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(y_test_s, y_pred_s, alpha=0.3, s=15, color='#2E5FA3')
lims = [0, min(2_000_000, max(y_test_s.max(), y_pred_s.max()))]
ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')
ax.set_xlim(lims); ax.set_ylim(lims)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1000:.0f}k'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1000:.0f}k'))
ax.set_xlabel('Actual Price')
ax.set_ylabel('Predicted Price')
ax.set_title(f'Actual vs Predicted Prices (R² = {r2:.3f})', fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Key Findings & Conclusions

---

### Summary of Results

| Finding | Detail |
|---|---|
| **Median house price (2023)** | See output above |
| **Most expensive property type** | Detached houses |
| **Most affordable property type** | Flats/Maisonettes |
| **New build premium** | New builds consistently priced higher across all types |
| **Most expensive county** | Typically Greater London / Surrey |
| **Most affordable county** | Northern counties (e.g. County Durham, Burnley area) |
| **Busiest sales month** | Typically late spring (May-June) |

### Key Insights

- **Regional inequality is stark.** Median prices in the most expensive counties can be 3-4 times those in the most affordable, reflecting the ongoing North-South divide in UK housing.
- **New builds carry a significant premium.** Across all property types, new build properties sell for noticeably more than equivalent existing properties - partly due to Help to Buy incentives and modern specifications.
- **Flats are the most accessible entry point** into homeownership but have seen slower appreciation compared to houses.
- **The linear model is a useful baseline** but confirms that property-level features (location, size, condition) are needed for accurate price prediction - a natural next step using gradient boosting or neural network approaches.

### Next Steps / Future Work

- Incorporate postcode-level data to add geographic precision
- Build a Random Forest or XGBoost model for improved prediction
- Extend the analysis across multiple years(1995-2024) to study long-term trends
- Map price data geographically using Folium or Geopandas

---

**Data source:** HM Land Registry Price Paid Data, licenced under the Open Government Licence v3.0.